# Capstone - Bakehouse End-to-End Analysis

**Dataset:** `samples.bakehouse` (all 6 tables)

**Difficulty:** Hard

**Topics:** multi-table joins, window functions, nested MAP output, `collect_set` with ranking, PII masking, summary stats loop, discount detection, complex aggregations

> These problems are intentionally open-ended - they combine multiple PySpark skills in a single question,
> the same way real-world data engineering tasks and technical assessments do.
> Each problem has one correct answer but many valid approaches.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_customers    = spark.table("samples.bakehouse.sales_customers")
df_franchises   = spark.table("samples.bakehouse.sales_franchises")
df_suppliers    = spark.table("samples.bakehouse.sales_suppliers")
df_reviews      = spark.table("samples.bakehouse.media_customer_reviews")

dataset_map = {
    "sales_transactions":        df_transactions,
    "sales_customers":           df_customers,
    "sales_franchises":          df_franchises,
    "sales_suppliers":           df_suppliers,
    "media_customer_reviews":    df_reviews,
}

## Problem 1 - Dataset Summary

For every table in the `dataset_map` above, compute a summary with:
- `dataset_name` - the key from `dataset_map`
- `row_count` - total number of rows
- `column_count` - total number of columns

Return a single DataFrame with one row per table, using `spark.createDataFrame()` to assemble the result.
Sort by `dataset_name`.

**Expected output columns:** `dataset_name`, `row_count`, `column_count`

In [ ]:
# Problem 1 - write your solution here
# Assign result to: result_1

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'dataset_name' in cols, "Missing column: dataset_name"
assert 'row_count' in cols, "Missing column: row_count"
assert 'column_count' in cols, "Missing column: column_count"
cnt = result_1.count()
assert cnt == 5, f"Expected 5 rows (one per table), got {cnt}"
min_rows = result_1.agg(F.min('row_count')).collect()[0][0]
assert min_rows > 0, "All tables should have at least 1 row"
min_cols = result_1.agg(F.min('column_count')).collect()[0][0]
assert min_cols >= 3, "All tables should have at least 3 columns"
print(f"Problem 1 passed ✓")
result_1.show()

## Problem 2 - Discount Detection

In `sales_transactions`, detect rows where a discount was applied.
A discount exists when `totalPrice < unitPrice * quantity`.

For each discounted transaction compute:
- `expected_price` = `unitPrice * quantity`
- `discount_amount` = `expected_price - totalPrice`
- `discountPercentage` = `(discount_amount / expected_price) * 100`

Return only the discounted rows, joined with `franchises` to include `franchiseID`.

**Expected output columns:** `transactionID`, `customerID`, `franchiseID`, `product`, `unitPrice`, `quantity`, `totalPrice`, `expected_price`, `discount_amount`, `discountPercentage`

In [ ]:
# Problem 2 - write your solution here
# Assign result to: result_2

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'discountpercentage' in cols or 'discount_percentage' in cols, "Missing column: discountPercentage"
assert 'expected_price' in cols, "Missing column: expected_price"
assert 'discount_amount' in cols, "Missing column: discount_amount"
cnt = result_2.count()
# Every row must genuinely have a discount
disc_col = [c for c in result_2.columns if 'discount_amount' in c.lower()][0]
non_discounted = result_2.filter(F.col(disc_col) <= 0).count()
assert non_discounted == 0, f"{non_discounted} rows have discount_amount <= 0 - filter not applied correctly"
print(f"Problem 2 passed ✓  ({cnt} discounted transactions found)")

## Problem 3 - Revenue per Franchise per Date

Join `sales_transactions` with `sales_franchises`.
Extract the date (no time component) from `dateTime`.
Aggregate by `date` and `franchiseID`:

- `quantitySold` = sum of quantity
- `revenue` = sum of totalPrice

Sort the result by `date` ascending, then `franchiseID` ascending.

**Expected output columns:** `date`, `franchiseID`, `franchise_name`, `quantitySold`, `revenue`

In [ ]:
# Problem 3 - write your solution here
# Assign result to: result_3

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'date' in cols, "Missing column: date"
assert 'franchiseid' in cols, "Missing column: franchiseID"
assert 'quantitysold' in cols or 'quantity_sold' in cols, "Missing column: quantitySold"
assert 'revenue' in cols, "Missing column: revenue"
cnt = result_3.count()
assert cnt > 0, "Got 0 rows"
# revenue must be positive
min_rev = result_3.agg(F.min('revenue')).collect()[0][0]
assert min_rev > 0, f"revenue must be > 0, got min={min_rev}"
# Check sort order - first date should be <= last date
dates = [r['date'] for r in result_3.select('date').limit(2).collect()]
print(f"Problem 3 passed ✓  ({cnt} franchise-date combinations)")

## Problem 4 - Top 3 Franchises per Day as a Nested Map

This is a multi-step problem combining window functions, aggregation, and complex types.

Starting from Problem 3's result (or rebuild it), for each `date`:

1. Rank franchises by revenue using a window function (`RANK() OVER (PARTITION BY date ORDER BY revenue DESC)`)
2. Keep only the top 3 (`rank <= 3`)
3. Aggregate each date into a single row where `franchises` is a **map** built with `map_from_entries`:
   - Each entry is a struct of `(franchiseName, revenueVolume, revenuePercentage)`
   - `revenuePercentage` = franchise revenue / total daily revenue × 100
   - `totalRevenue` = total revenue for that day across ALL franchises (not just the top 3)

**Expected output columns:** `date`, `totalRevenue`, `franchises` (MapType)

> Break this into steps: (1) rank, (2) compute daily total, (3) filter top 3, (4) collect into map.

In [ ]:
# Problem 4 - write your solution here
# Assign result to: result_4

result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'date' in cols, "Missing column: date"
assert 'totalrevenue' in cols or 'total_revenue' in cols, "Missing column: totalRevenue"
assert 'franchises' in cols, "Missing column: franchises (should be a MapType)"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
# One row per date
distinct_dates = result_4.select('date').distinct().count()
assert cnt == distinct_dates, f"Should have exactly one row per date. Got {cnt} rows but {distinct_dates} distinct dates"
# franchises column should be MapType
franchises_field = [f for f in result_4.schema.fields if f.name == 'franchises'][0]
assert isinstance(franchises_field.dataType, T.MapType), "franchises column must be MapType"
print(f"Problem 4 passed ✓  ({cnt} dates, each with top-3 franchises as a map)")

## Problem 5 - Customer Behaviour by Country and Store Size

Join `sales_transactions` + `sales_customers` + `sales_franchises`.
Per `country` (from franchises) and `franchiseSize`:

- `numberOfUniqueCustomers` = count of distinct `customerID`s
- `averageBasket` = average number of distinct products per customer per visit (transaction)

Sort by `country` and `franchiseSize`.

**Expected output columns:** `country`, `franchiseSize`, `numberOfUniqueCustomers`, `averageBasket`

In [ ]:
# Problem 5 - write your solution here
# Assign result to: result_5

result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'country' in cols, "Missing column: country"
assert 'franchisesize' in cols or 'franchise_size' in cols, "Missing column: franchiseSize"
assert 'numberofuniquecustomers' in cols or 'number_of_unique_customers' in cols or 'numberofluniquecustomers' in [c.lower().replace(' ','') for c in result_5.columns], "Missing column: numberOfUniqueCustomers"
assert 'averagebasket' in cols or 'average_basket' in cols, "Missing column: averageBasket"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
cust_col = [c for c in result_5.columns if 'unique' in c.lower() or 'customer' in c.lower()][0]
min_cust = result_5.agg(F.min(cust_col)).collect()[0][0]
assert min_cust >= 1, "numberOfUniqueCustomers must be >= 1"
print(f"Problem 5 passed ✓  ({cnt} country-size combinations)")

## Problem 6 - Top 3 Products per Franchise with Countries

Join `sales_transactions` + `sales_franchises` + `sales_customers` (to get customer country).

1. Compute total revenue per (`franchiseID`, `franchiseName`, `product`)
2. Use a window function to rank products within each franchise by revenue (rank 1 = highest)
3. Keep only `rank <= 3`
4. Use `collect_set` on customer country to build a `countries` array showing where that product is sold

**Expected output columns:** `franchiseName`, `product`, `revenue`, `countries` (ArrayType)

In [ ]:
# Problem 6 - write your solution here
# Assign result to: result_6

result_6 = None  # replace this

In [ ]:
# ── Tests for Problem 6 ──────────────────────────────────────────
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_6.columns]
assert 'franchisename' in cols or 'franchise_name' in cols, "Missing column: franchiseName"
assert 'product' in cols, "Missing column: product"
assert 'revenue' in cols, "Missing column: revenue"
assert 'countries' in cols, "Missing column: countries (should be ArrayType)"
cnt = result_6.count()
assert cnt > 0, "Got 0 rows"
# countries must be ArrayType
countries_field = [f for f in result_6.schema.fields if f.name.lower() == 'countries'][0]
assert isinstance(countries_field.dataType, T.ArrayType), "countries must be ArrayType"
# Each franchise should have at most 3 products
fname_col = [c for c in result_6.columns if 'franchise' in c.lower() and 'name' in c.lower()][0]
max_products = result_6.groupBy(fname_col).count().agg(F.max('count')).collect()[0][0]
assert max_products <= 3, f"Each franchise should have at most 3 products, got {max_products}"
print(f"Problem 6 passed ✓  ({cnt} franchise-product rows, max {max_products} products per franchise)")

## Problem 7 - Franchise Reviews with Sentiment

Join `media_customer_reviews` with `sales_franchises` on `franchiseID`.

1. Count total reviews per franchise (`numberReviews`)
2. Use `ai_analyze_sentiment` (via `spark.sql`) to classify each review's sentiment
3. Aggregate: per franchise per sentiment, count reviews and compute `percentageOfReviews` = count / total × 100

Sort by `numberReviews` descending.

**Expected output columns:** `franchiseName`, `country`, `sentiment`, `numberReviews`, `percentageOfReviews`

> If `ai_analyze_sentiment` is not available on your cluster, use a placeholder:
> `F.when(F.length('review') > 100, 'positive').otherwise('neutral').alias('sentiment')`

In [ ]:
# Problem 7 - write your solution here
# Assign result to: result_7

result_7 = None  # replace this

In [ ]:
# ── Tests for Problem 7 ──────────────────────────────────────────
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_7.columns]
assert 'sentiment' in cols, "Missing column: sentiment"
assert 'numberreviews' in cols or 'number_reviews' in cols, "Missing column: numberReviews"
assert any('percentage' in c or 'pct' in c for c in cols), "Missing: percentageOfReviews"
cnt = result_7.count()
assert cnt > 0, "Got 0 rows"
pct_col = [c for c in result_7.columns if 'percentage' in c.lower() or 'pct' in c.lower()][0]
max_pct = result_7.agg(F.max(pct_col)).collect()[0][0]
assert max_pct <= 100.0, f"percentageOfReviews should be <= 100, got {max_pct}"
print(f"Problem 7 passed ✓  ({cnt} franchise-sentiment rows)")

## Problem 8 - Full PII-Masked Operational Dashboard

Join all four sales tables: `sales_transactions` + `sales_customers` + `sales_franchises`.
Build a single flat dataset for an operational dashboard where all PII is masked:

| Column | Masking rule |
|---|---|
| `firstName` | Replace all characters with `*` |
| `lastName` | Replace all characters with `*` |
| `emailAddress` | Mask username (keep first char + `***`), keep `@domain` intact |
| `phoneNumber` | Keep last 3 digits, replace all others with `*` |
| `cardNumber` | Cast to string, keep last 4 digits, replace rest with `*` |

**Expected output columns:** `date`, `dateTime`, `franchiseName`, `firstName`, `lastName`, `emailAddress`, `phoneNumber`, `city`, `country`, `product`, `quantity`, `unitPrice`, `totalPrice`, `methodOfPayment`, `cardNumber`

In [ ]:
# Problem 8 - write your solution here
# Assign result to: result_8

result_8 = None  # replace this

In [ ]:
# ── Tests for Problem 8 ──────────────────────────────────────────
assert result_8 is not None, "result_8 is None"
assert hasattr(result_8, 'columns'), "Must be a Spark DataFrame"
cols_lower = [c.lower() for c in result_8.columns]
for required in ['date', 'datetime', 'franchisename', 'firstname', 'lastname',
                 'emailaddress', 'phonenumber', 'city', 'country', 'product',
                 'quantity', 'unitprice', 'totalprice', 'methodofpayment', 'cardnumber']:
    assert required in cols_lower, f"Missing column: {required}"
cnt = result_8.count()
assert cnt > 0, "Got 0 rows"
# Verify masking: firstName should contain only * characters
fn_col = [c for c in result_8.columns if c.lower() == 'firstname'][0]
unmasked_names = result_8.filter(F.col(fn_col).rlike(r'[a-zA-Z]')).count()
assert unmasked_names == 0, f"firstName still contains letters - masking incomplete ({unmasked_names} rows)"
# Email should still contain @ symbol
em_col = [c for c in result_8.columns if c.lower() == 'emailaddress'][0]
no_at = result_8.filter(~F.col(em_col).contains('@')).count()
assert no_at == 0, f"{no_at} emails are missing @ symbol - domain was removed"
# cardNumber should not contain digits except last 4
card_col = [c for c in result_8.columns if c.lower() == 'cardnumber'][0]
sample_card = result_8.select(card_col).first()[0]
if sample_card:
    assert str(sample_card).count('*') >= 4, f"cardNumber masking looks incorrect: {sample_card}"
print(f"Problem 8 passed ✓  ({cnt} rows, PII masking verified)")